# Лабораторная работа №2. Проведение исследований с логистической и линейной регрессией

**Студент:** Чибугаев Илья Алексеевич  
**Вариант:** 22  
**Случайное зерно:** 22

## Исходные данные

- **Классификация:** [Ionosphere, UCI](https://archive.ics.uci.edu/dataset/52/ionosphere) — 351 объект, 34 числовых признака; `bad=1` обозначает сигнал без структуры в ионосфере.
- **Регрессия:** [Concrete Compressive Strength, UCI](https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength) — 1030 составов бетона, 8 признаков, целевая переменная — прочность в МПа.

Датасеты выбраны для разных практических задач: контроль качества радиосигнала и предварительная оценка рецептуры бетона. Оба набора небольшие, содержат только числовые признаки и позволяют сопоставить алгоритмы без изменения задачи между лабораторными.

Логистическая регрессия оценивает вероятность класса через сигмоиду линейной комбинации признаков. Линейная регрессия минимизирует сумму квадратов ошибок; полиномиальные признаки позволяют ей описывать нелинейность в исходных переменных.

## Протокол и метрики

Для обеих задач используется фиксированное разбиение **80% / 20%** с `random_state=22`; в классификации разбиение стратифицировано. Подбор параметров выполняется только внутри обучающей выборки. Все четыре конфигурации (sklearn / собственная, базовая / улучшенная) оцениваются на одном и том же тестовом наборе.

- **Классификация:** основная метрика — **ROC-AUC**, поскольку она оценивает ранжирование и не зависит от одного порога; **F1** контролирует качество по классу `bad`, а **accuracy** показывает общую долю верных ответов.
- **Регрессия:** основная метрика — **RMSE (МПа)**, штрафующая крупные ошибки; **MAE (МПа)** даёт типичную абсолютную ошибку, **R²** — долю объяснённой дисперсии.


## 1. Подготовка

Код импортирует только общие функции и фиксирует настройки отображения. Каждая следующая ячейка имеет отдельный комментарий.

In [1]:
# Настраиваем воспроизводимое окружение и импортируем общий код эксперимента.
from pathlib import Path
import sys
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.experiments import TaskExperiment, LAB_NAMES

pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 4)
plt.style.use("seaborn-v0_8-whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)
print(f"Корень проекта: {ROOT}")

Корень проекта: /Users/finleyl/Documents/kimi/tasks/2026-11-10/20-02-04-6933ea61/ml-labs-1-5


### Загрузка и проверка разбиений

Проверяем размеры, отсутствие пропусков, баланс классов и диапазон целевой переменной. Разбиения одинаковы во всех пяти работах.

In [2]:
# Создаём эксперименты для двух задач и выводим контрольные характеристики данных.
experiments = {task: TaskExperiment(2, task) for task in ("classification", "regression")}
for task, experiment in experiments.items():
    data = experiment.data
    print(f"{task}: {data.name}")
    print(data.practical_task)
    print(f"train={data.X_train.shape}, test={data.X_test.shape}, пропуски={int(data.X.isna().sum().sum())}")
    if task == "classification":
        print(f"Доля bad: train={data.y_train.mean():.3f}, test={data.y_test.mean():.3f}")
    else:
        print(data.y.describe().to_string())
    print()

classification: Ionosphere (UCI)
По сигналам фазированной антенной решётки отличать радиосигналы со структурой в ионосфере от проходящих сигналов.
train=(280, 34), test=(71, 34), пропуски=0
Доля bad: train=0.361, test=0.352

regression: Concrete Compressive Strength (UCI)
По составу смеси и возрасту бетона прогнозировать прочность на сжатие в МПа для предварительной оценки рецептуры.
train=(824, 8), test=(206, 8), пропуски=0
count    1030.0000
mean       35.8178
std        16.7057
min         2.3318
25%        23.7071
50%        34.4428
75%        46.1363
max        82.5992



## 2. Базовый бейзлайн sklearn

Бейзлайн использует параметры алгоритма по умолчанию и только медианное заполнение пропусков. Это точка отсчёта, а не ожидаемый победитель.

In [3]:
# Обучаем базовые sklearn-модели и оцениваем их на отложенной выборке.
for experiment in experiments.values():
    experiment.fit_sklearn_baseline()
baseline_results = pd.concat([experiment.results for experiment in experiments.values()], ignore_index=True)
display(baseline_results.drop(columns=["params"]))

classification | sklearn | baseline: accuracy=0.7746, f1=0.5294, roc_auc=0.8313; 0.00 s
regression | sklearn | baseline: rmse=10.1625, mae=8.1968, r2=0.6542; 0.00 s


,lab,algorithm,task,implementation,stage,accuracy,f1,roc_auc,rmse,mae,r2,fit_seconds
0,2,Логистическая / линейная регрессия,classification,sklearn,baseline,0.7746,0.5294,0.8313,NaN,NaN,NaN,0.0037
1,2,Логистическая / линейная регрессия,regression,sklearn,baseline,NaN,NaN,NaN,10.1625,8.1968,0.6542,0.0014


## 3. Улучшение бейзлайна

### Проверяемые гипотезы

1. Стандартизация и регуляризация улучшат устойчивость логистической регрессии при коррелированных сигналах.
2. Балансировка классов проверяется как способ повысить качество распознавания редкого класса bad, но выбор делается по ROC-AUC на CV.
3. Для бетона проверяются попарные взаимодействия и квадраты признаков: зависимость прочности от состава и возраста нелинейна.

Метрика подбора — ROC-AUC для классификации и RMSE для регрессии. Лучшие параметры затем один раз оцениваются на тесте.

In [4]:
# Запускаем GridSearchCV и показываем пять лучших конфигураций для каждой задачи.
for experiment in experiments.values():
    experiment.fit_sklearn_improved()
    print(f"{experiment.task}: лучшие параметры = {experiment.search.best_params_}")
    display(experiment.cv_table().head())

classification | sklearn | improved: accuracy=0.8028, f1=0.6316, roc_auc=0.8696; 1.31 s
classification: лучшие параметры = {'model__C': 0.2, 'model__class_weight': 'balanced'}


,params,cv_roc_auc,std_roc_auc,mean_fit_time
0,"{'model__C': 0.2, 'model__class_weight': 'bala...",0.9375,0.0061,0.0053
1,"{'model__C': 0.2, 'model__class_weight': None}",0.9353,0.0077,0.0154
2,"{'model__C': 0.05, 'model__class_weight': 'bal...",0.9320,0.0145,0.0225
3,"{'model__C': 0.05, 'model__class_weight': None}",0.9312,0.0162,0.0207
4,"{'model__C': 1.0, 'model__class_weight': None}",0.9208,0.0279,0.0061


regression | sklearn | improved: rmse=7.5310, mae=5.8169, r2=0.8101; 0.03 s
regression: лучшие параметры = {'poly__degree': 2}


,params,cv_rmse,std_rmse,mean_fit_time
0,{'poly__degree': 2},7.7647,0.4435,0.0031
1,{'poly__degree': 1},10.5173,0.4121,0.0023


### Сравнение базового и улучшенного sklearn-бейзлайна

Положительная разница ROC-AUC/F1/accuracy/R² означает улучшение; для RMSE/MAE улучшение соответствует отрицательной разнице.

In [5]:
# Формируем таблицу изменений метрик между улучшенной и базовой sklearn-моделями.
metric_names = {"classification": ["roc_auc", "f1", "accuracy"], "regression": ["rmse", "mae", "r2"]}
for task, experiment in experiments.items():
    rows = experiment.results.set_index("stage")
    metrics = metric_names[task]
    comparison = rows.loc[["baseline", "improved"], metrics].T
    comparison["change"] = comparison["improved"] - comparison["baseline"]
    print(task)
    display(comparison)

classification


stage,baseline,improved,change
roc_auc,0.8313,0.8696,0.0383
f1,0.5294,0.6316,0.1022
accuracy,0.7746,0.8028,0.0282


regression


stage,baseline,improved,change
rmse,10.1625,7.5310,-2.6315
mae,8.1968,5.8169,-2.3800
r2,0.6542,0.8101,0.1559


## 4. Собственная реализация

Логистическая регрессия обучается полным градиентным спуском со стабильной сигмоидой и L2-регуляризацией. Линейная регрессия использует np.linalg.lstsq вместо явного обращения матрицы, что устойчивее при мультиколлинеарности.

Исходный код находится в `src/models/`. Сначала используются базовые параметры, затем техники и параметры улучшенного бейзлайна.

In [6]:
# Обучаем собственные базовые модели на тех же преобразованных данных, что и sklearn-бейзлайн.
for experiment in experiments.values():
    experiment.fit_custom_baseline()
custom_baseline_results = pd.concat([
    experiment.results[experiment.results["implementation"] == "from_scratch"]
    for experiment in experiments.values()
], ignore_index=True)
display(custom_baseline_results.drop(columns=["params"]))

classification | from_scratch | baseline: accuracy=0.7606, f1=0.4848, roc_auc=0.6730; 0.06 s
regression | from_scratch | baseline: rmse=10.1625, mae=8.1968, r2=0.6542; 0.00 s


,lab,algorithm,task,implementation,stage,accuracy,f1,roc_auc,rmse,mae,r2,fit_seconds
0,2,Логистическая / линейная регрессия,classification,from_scratch,baseline,0.7606,0.4848,0.673,NaN,NaN,NaN,0.0639
1,2,Логистическая / линейная регрессия,regression,from_scratch,baseline,NaN,NaN,NaN,10.1625,8.1968,0.6542,0.0001


### Собственная модель с улучшенным бейзлайном

Преобразования обучены только на train. Для собственной модели переносятся выбранные на CV параметры; исключение по числу деревьев случайного леса явно указано в гипотезах.

In [7]:
# Добавляем улучшенный препроцессинг и выбранные параметры в собственные реализации.
for experiment in experiments.values():
    experiment.fit_custom_improved()
all_lab_results = pd.concat([experiment.results for experiment in experiments.values()], ignore_index=True)
display(all_lab_results.drop(columns=["params"]))

classification | from_scratch | improved: accuracy=0.8028, f1=0.6316, roc_auc=0.8704; 0.03 s
regression | from_scratch | improved: rmse=7.5310, mae=5.8169, r2=0.8101; 0.00 s


,lab,algorithm,task,implementation,stage,accuracy,f1,roc_auc,rmse,mae,r2,fit_seconds
0,2,Логистическая / линейная регрессия,classification,sklearn,baseline,0.7746,0.5294,0.8313,NaN,NaN,NaN,0.0037
1,2,Логистическая / линейная регрессия,classification,sklearn,improved,0.8028,0.6316,0.8696,NaN,NaN,NaN,1.3052
2,2,Логистическая / линейная регрессия,classification,from_scratch,baseline,0.7606,0.4848,0.6730,NaN,NaN,NaN,0.0639
3,2,Логистическая / линейная регрессия,classification,from_scratch,improved,0.8028,0.6316,0.8704,NaN,NaN,NaN,0.0302
4,2,Логистическая / линейная регрессия,regression,sklearn,baseline,NaN,NaN,NaN,10.1625,8.1968,0.6542,0.0014
5,2,Логистическая / линейная регрессия,regression,sklearn,improved,NaN,NaN,NaN,7.5310,5.8169,0.8101,0.0305
6,2,Логистическая / линейная регрессия,regression,from_scratch,baseline,NaN,NaN,NaN,10.1625,8.1968,0.6542,0.0001
7,2,Логистическая / линейная регрессия,regression,from_scratch,improved,NaN,NaN,NaN,7.5310,5.8169,0.8101,0.0020


## 6. Сохранение результатов и выводы

Вывод формируется по фактическим значениям, а не по предположению, что улучшение обязано помочь. На маленькой тестовой выборке небольшие различия не следует трактовать как статистически доказанное превосходство.

In [8]:
# Сохраняем четыре результата на задачу и печатаем аргументированное заключение по каждой задаче.
output_path = ROOT / "results" / "lab02_results.csv"
output_path.parent.mkdir(exist_ok=True)
all_lab_results.to_csv(output_path, index=False)

for task, experiment in experiments.items():
    frame = experiment.results.set_index(["implementation", "stage"])
    metrics = metric_names[task]
    sklearn_base = frame.loc[("sklearn", "baseline"), metrics]
    sklearn_best = frame.loc[("sklearn", "improved"), metrics]
    custom_base = frame.loc[("from_scratch", "baseline"), metrics]
    custom_best = frame.loc[("from_scratch", "improved"), metrics]
    primary = "roc_auc" if task == "classification" else "rmse"
    higher_is_better = primary != "rmse"
    sklearn_change = sklearn_best[primary] - sklearn_base[primary]
    custom_change = custom_best[primary] - custom_base[primary]
    sklearn_verdict = "улучшился" if (sklearn_change > 0) == higher_is_better else "не улучшился"
    custom_verdict = "улучшился" if (custom_change > 0) == higher_is_better else "не улучшился"
    print(f"{task}: sklearn {primary} {sklearn_base[primary]:.4f} → {sklearn_best[primary]:.4f} ({sklearn_verdict}); "
          f"своя реализация {custom_base[primary]:.4f} → {custom_best[primary]:.4f} ({custom_verdict}).")
    print(f"Лучшие параметры sklearn: {experiment.search.best_params_}")
print(f"Результаты сохранены: {output_path.relative_to(ROOT)}")

classification: sklearn roc_auc 0.8313 → 0.8696 (улучшился); своя реализация 0.6730 → 0.8704 (улучшился).
Лучшие параметры sklearn: {'model__C': 0.2, 'model__class_weight': 'balanced'}
regression: sklearn rmse 10.1625 → 7.5310 (улучшился); своя реализация 10.1625 → 7.5310 (улучшился).
Лучшие параметры sklearn: {'poly__degree': 2}
Результаты сохранены: results/lab02_results.csv
